# 05c · Anexo de robustez R.a: sustento de no diferenciar (16 series)

**Iteración 3b, adenda del 26-oct-2026** (§2, anexo R; tarea DX). Por D15, las series transformadas
pasan a un anexo de robustez del Rmd final. Este notebook es la versión en Python (paridad, D6) de
la parte **R.a** del anexo de `R/ciclo_financiero_final.Rmd`:

1. **Pasos de la transformación:** PCA de las series sin transformar, solo con logaritmo, solo con la
   diferencia simple y con la transformación completa (inercia PC1–PC4, Horn, |r(PC1, t)| y
   congruencia de Tucker con PC1–PC3 sin transformar).
2. **Ganancia** del filtro de primera diferencia, $2\sin(\pi/P)$, en los períodos de interés.
3. **Espectro:** parte de la varianza de cada serie en oscilaciones de 24 meses o más
   (periodograma sin ventana, con el relleno de `spec.pgram` hasta `nextn(n)`).
4. **Estacionalidad residual:** ACF a 1 y 12 meses de las series transformadas.

La parte **R.b** (detección de episodios fuera de muestra) espera D17 y no se calcula aquí.

**Parámetros** (los mismos del 05): `EXCLUIDAS` (D16) y `CARPETA`; las salidas van en
`CARPETA/anexo/05c_*.csv`. **Orden:** primero el Rmd final (escribe los `R_anexo_*.csv`), luego este
notebook (los compara) y de nuevo el Rmd (paridad).

**Semillas.** Horn con 500 simulaciones y `numpy.default_rng(2026)`; el Rmd usa la misma cifra con el
generador de R. Rige la excepción de generadores: se compara el número de componentes retenidos, no
el umbral. Nada más es aleatorio.

In [1]:
import json
import os
import sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

from src.datos import META as META_DATOS, EXCLUIDAS_D16, MD5_D4, meta_vigente, _md5
from src.multivariado import estandarizar, pca_correlacion, horn, tucker
from src.series_tiempo import acf, quitar_tendencia

# ---- Parámetros (adenda 3b, §3) ----------------------------------------------------------
EXCLUIDAS = tuple(json.loads(os.environ['TESIS_EXCLUIDAS'])) if 'TESIS_EXCLUIDAS' in os.environ else EXCLUIDAS_D16
CARPETA = os.environ.get('TESIS_CARPETA', 'data/results/final')
DIR = os.path.join(CARPETA, 'anexo')
os.makedirs(DIR, exist_ok=True)
assert os.path.normpath(DIR) not in (os.path.normpath('data/results'), os.path.normpath('data/results/niveles'))
SEMILLA_HORN = 2026        # params$semilla del Rmd
N_SIM_HORN = 500
PERIODOS = [6, 12, 24, 36, 60, 96, 120, 228, 360]
P_MIN = 24                 # meses: «oscilaciones de dos años o más»
print('EXCLUIDAS =', EXCLUIDAS, '| salidas en', DIR)

EXCLUIDAS = ('Liquidez M2',) | salidas en data/results/final/anexo


## 1. Datos

Las mismas series y el mismo orden que la sección 2 del Rmd final (sin las excluidas por D16). Se
comprueban los md5 de D4, el calendario, la ausencia de faltantes y que las transformadas se
reconstruyen desde los niveles.

In [2]:
for nombre, h in MD5_D4.items():
    assert _md5(os.path.join('data/processed', nombre)) == h, f'{nombre} no es la versión congelada (D4)'
META = meta_vigente(EXCLUIDAS)
COLUMNAS, NOMBRES = list(META.columna), list(META.nombre_corto)
P = len(COLUMNAS)
niv = pd.read_csv('data/processed/dataset_niveles_mensual.csv', parse_dates=['fecha'])
tra = pd.read_csv('data/processed/dataset_transformado_mensual.csv', parse_dates=['fecha'])
base_tra = {c.replace('_log_diff', '').replace('_diff', ''): c for c in tra.columns if c != 'fecha'}
assert set(base_tra) == set(META_DATOS.columna)
es_log = np.array([base_tra[c].endswith('_log_diff') for c in COLUMNAS])
X = niv[COLUMNAS].to_numpy(dtype=float)
Xt = tra[[base_tra[c] for c in COLUMNAS]].to_numpy(dtype=float)
for f in (niv['fecha'], tra['fecha']):
    esperado = pd.date_range(f.iloc[0], f.iloc[-1], freq='MS')
    assert len(esperado) == len(f) and (esperado == pd.DatetimeIndex(f)).all(), 'calendario con huecos'
assert X.shape == (228, P) and Xt.shape == (227, P) and not np.isnan(X).any() and not np.isnan(Xt).any()
rec = np.column_stack([np.diff(np.log(X[:, j])) if es_log[j] else np.diff(X[:, j]) for j in range(P)])
dif_transf = np.abs(rec - Xt).max()
assert dif_transf < 1e-8
print(f'{X.shape[0]} × {P} niveles; {Xt.shape[0]} × {P} transformadas; reconstrucción: {dif_transf:.2e}')

228 × 16 niveles; 227 × 16 transformadas; reconstrucción: 5.68e-14


## 2. Qué paso cambia la estructura (chunk `ra-pasos`)

Para cada versión: PCA de la matriz de correlaciones con la regla de signo de Rcamiz
(`pca_correlacion`), inercia (%) de PC1–PC4, componentes que retiene Horn (p95, 500 simulaciones),
|r| entre el score de PC1 y el número de mes, y congruencia de Tucker de PC1–PC3 sin transformar con
la componente más parecida entre las seis primeras de la versión (valor absoluto, sin emparejar).

In [3]:
logX = X.copy()
logX[:, es_log] = np.log(X[:, es_log])
versiones = {'sin_transformar': X, 'solo_logaritmo': logX, 'solo_diferencia': np.diff(X, axis=0), 'log_diferencia': Xt}


def version_pca(M):
    Zm = estandarizar(M)
    pc = pca_correlacion(Zm)
    t = np.arange(1, M.shape[0] + 1)
    return {'inercia': 100 * pc['valores'] / M.shape[1], 'cargas': pc['cargas'],
            'horn': horn(Zm, n_sim=N_SIM_HORN, semilla=SEMILLA_HORN)['retener'],
            'r_t': float(np.corrcoef(pc['scores'][:, 0], t)[0, 1])}


vers = {v: version_pca(M) for v, M in versiones.items()}
ref = vers['sin_transformar']['cargas']
filas = []
for v, r in vers.items():
    for k in range(4):
        filas.append({'version': v, 'medida': 'inercia', 'componente': k + 1, 'valor': r['inercia'][k]})
    filas.append({'version': v, 'medida': 'horn_retenidas', 'componente': pd.NA, 'valor': float(r['horn'])})
    filas.append({'version': v, 'medida': 'r_pc1_tiempo_abs', 'componente': pd.NA, 'valor': abs(r['r_t'])})
    for k in range(3):
        cg = max(abs(tucker(ref[:, k], r['cargas'][:, j])) for j in range(6))
        filas.append({'version': v, 'medida': 'congruencia_con_sin_transformar', 'componente': k + 1, 'valor': cg})
pasos = pd.DataFrame(filas)
pasos['componente'] = pasos['componente'].astype('Int64')
pasos.to_csv(os.path.join(DIR, '05c_pasos.csv'), index=False)
print(pasos.round(3).to_string(index=False))

        version                          medida  componente  valor
sin_transformar                         inercia           1 65.228
sin_transformar                         inercia           2 19.382
sin_transformar                         inercia           3  7.552
sin_transformar                         inercia           4  3.075
sin_transformar                  horn_retenidas        <NA>  2.000
sin_transformar                r_pc1_tiempo_abs        <NA>  0.993
sin_transformar congruencia_con_sin_transformar           1  1.000
sin_transformar congruencia_con_sin_transformar           2  1.000
sin_transformar congruencia_con_sin_transformar           3  1.000
 solo_logaritmo                         inercia           1 65.197
 solo_logaritmo                         inercia           2 18.764
 solo_logaritmo                         inercia           3  8.367
 solo_logaritmo                         inercia           4  3.683
 solo_logaritmo                  horn_retenidas        <NA>  2

## 3. Ganancia de la primera diferencia (chunk `ra-ganancia`)

$G(\omega) = |1 - e^{-i\omega}| = 2\sin(\omega/2) = 2\sin(\pi/P)$ para un período de $P$ meses; la
varianza que conserva la oscilación es $G^2$. Resultado exacto, que no depende de los datos.

In [4]:
gan = pd.DataFrame({'periodo_meses': PERIODOS, 'ganancia': 2 * np.sin(np.pi / np.array(PERIODOS, dtype=float))})
gan['varianza_conservada'] = gan['ganancia'] ** 2
gan.to_csv(os.path.join(DIR, '05c_ganancia.csv'), index=False)
print(gan.round(4).to_string(index=False))

 periodo_meses  ganancia  varianza_conservada
             6    1.0000               1.0000
            12    0.5176               0.2679
            24    0.2611               0.0681
            36    0.1743               0.0304
            60    0.1047               0.0110
            96    0.0654               0.0043
           120    0.0524               0.0027
           228    0.0276               0.0008
           360    0.0175               0.0003


## 4. Varianza en oscilaciones de 24 meses o más (chunk `ra-espectro`)

Traducción de `spec.pgram(x - mean(x), taper = 0, detrend = FALSE)` de R: con `fast = TRUE` (valor
por defecto) la serie se rellena con ceros hasta `nextn(n)`, el menor entero ≥ n cuyos factores
primos son 2, 3 y 5; el periodograma se evalúa en las frecuencias $k/N$, $k = 1, \dots, \lfloor N/2
\rfloor$. La fracción lenta es la suma de las ordenadas con $1/f \ge 24$ sobre la suma total (las
constantes de escala se cancelan).

In [5]:
def nextn(n, factores=(2, 3, 5)):
    """Menor entero >= n que solo tiene los factores primos dados (stats::nextn de R)."""
    m = n
    while True:
        r = m
        for f in factores:
            while r % f == 0:
                r //= f
        if r == 1:
            return m
        m += 1


def frac_lenta(x, p_min=P_MIN):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    N = nextn(len(x))
    I = np.abs(np.fft.fft(np.concatenate([x, np.zeros(N - len(x))]))) ** 2
    k = np.arange(1, N // 2 + 1)
    espec = I[k]
    return float(espec[(N / k) >= p_min].sum() / espec.sum())


assert nextn(228) == 240 and nextn(227) == 240 and nextn(240) == 240
espectro = pd.DataFrame({'serie': NOMBRES,
                         'niv': [frac_lenta(X[:, j]) for j in range(P)],
                         'niv_det': [frac_lenta(quitar_tendencia(X[:, [j]])[:, 0]) for j in range(P)],
                         'tr': [frac_lenta(Xt[:, j]) for j in range(P)]})
espectro.to_csv(os.path.join(DIR, '05c_espectro.csv'), index=False)
print((espectro.set_index('serie') * 100).round(1).to_string())
print('Medianas (%):', (100 * espectro[['niv', 'niv_det', 'tr']].median()).round(1).to_dict())

                                 niv  niv_det    tr
serie                                              
Crédito empresarial             97.4     93.9  31.3
Crédito consumo                 97.3     95.4  38.8
Crédito hipotecario             97.3     91.0  36.7
M1 (dinero)                     96.9     92.6  10.7
M3 (liquidez total)             97.3     94.4  18.9
Tasa de referencia              97.4     97.3  57.2
Tasa interbancaria              97.1     97.0  54.1
TAMN                            97.1     94.1  30.8
TIPMN                           96.6     96.9  61.4
IPC Lima                        97.2     98.1  24.7
Tipo de cambio                  97.8     89.2  16.6
Índice BVL                      83.2     78.5  10.1
EMBIG Perú                      71.1     69.0   7.6
PBI desestacionalizado          93.0     57.6   2.6
Demanda interna                 89.7     47.4   1.1
Reservas internacionales netas  94.8     92.8  22.1
Medianas (%): {'niv': 97.1, 'niv_det': 93.4, 'tr': 23.4}


## 5. Estacionalidad residual (chunk `ra-estacional`)

ACF a 1 y 12 meses de las series transformadas, con el estimador de `stats::acf` (serie centrada,
divisor $n$); banda aproximada $\pm 2/\sqrt{n}$.

In [6]:
estac = pd.DataFrame({'serie': NOMBRES, 'acf1': [acf(Xt[:, j], [1])[0] for j in range(P)],
                      'acf12': [acf(Xt[:, j], [12])[0] for j in range(P)]})
estac.to_csv(os.path.join(DIR, '05c_estacional.csv'), index=False)
umbral = 2 / np.sqrt(Xt.shape[0])
print(f'Banda ±2/√n = {umbral:.3f}; series con ACF a 12 meses sobre la banda: {int((estac.acf12 > umbral).sum())} de {P}')
print(estac.sort_values('acf12', ascending=False).round(3).to_string(index=False))

Banda ±2/√n = 0.133; series con ACF a 12 meses sobre la banda: 7 de 16
                         serie   acf1  acf12
               Demanda interna -0.194  0.580
                   M1 (dinero) -0.086  0.388
           Crédito empresarial  0.381  0.341
                      IPC Lima  0.276  0.318
           Crédito hipotecario  0.255  0.290
               Crédito consumo  0.308  0.171
           M3 (liquidez total)  0.126  0.141
                          TAMN  0.351  0.093
        PBI desestacionalizado  0.306 -0.013
                Tipo de cambio  0.241 -0.016
Reservas internacionales netas  0.166 -0.017
                    Índice BVL  0.146 -0.041
                         TIPMN  0.750 -0.075
            Tasa de referencia  0.730 -0.093
                    EMBIG Perú  0.154 -0.104
            Tasa interbancaria  0.689 -0.107


## 6. Comprobaciones contra los archivos del Rmd

Las filas oficiales de paridad las escribe el Rmd en `CARPETA/anexo/R_paridad_python.csv`. Aquí se
comprueba en el sentido inverso (diferencia relativa |a − b| / max(1, |b|)). Horn: se compara el
número de componentes retenidos (excepción de generadores).

In [7]:
def dif_max(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.max(np.abs(a - b) / np.maximum(1, np.abs(b))))


chequeos = []
r = pd.read_csv(os.path.join(DIR, 'R_anexo_pasos.csv'))
m = pasos.merge(r, on=['version', 'medida', 'componente'], suffixes=('_py', '_r'))
assert len(m) == len(pasos) == len(r)
h = m['medida'] == 'horn_retenidas'
chequeos.append(('pasos: inercia, |r(PC1, t)| y congruencias', dif_max(m.loc[~h, 'valor_py'], m.loc[~h, 'valor_r']), 1e-10))
chequeos.append(('pasos: componentes retenidos por Horn (número)', dif_max(m.loc[h, 'valor_py'], m.loc[h, 'valor_r']), 0))
r = pd.read_csv(os.path.join(DIR, 'R_anexo_ganancia.csv'))
chequeos.append(('ganancia', dif_max(gan[['ganancia', 'varianza_conservada']], r[['ganancia', 'varianza_conservada']]), 1e-14))
r = pd.read_csv(os.path.join(DIR, 'R_anexo_espectro.csv'))
m = espectro.merge(r, on='serie', suffixes=('_py', '_r'))
assert len(m) == P
chequeos.append(('espectro: fracción en períodos >= 24 meses', max(dif_max(m[c + '_py'], m[c + '_r']) for c in ['niv', 'niv_det', 'tr']), 1e-10))
r = pd.read_csv(os.path.join(DIR, 'R_anexo_estacional.csv'))
m = estac.merge(r, on='serie', suffixes=('_py', '_r'))
assert len(m) == P
chequeos.append(('estacionalidad: ACF a 1 y 12 meses', max(dif_max(m[c + '_py'], m[c + '_r']) for c in ['acf1', 'acf12']), 1e-10))
tabla = pd.DataFrame(chequeos, columns=['comprobacion', 'dif_max', 'tolerancia'])
tabla['coincide'] = tabla['dif_max'] <= tabla['tolerancia']
print(tabla.to_string(index=False))
assert tabla['coincide'].all(), 'alguna comprobación contra el Rmd no coincide'
print('Archivos escritos:', ', '.join(sorted(f for f in os.listdir(DIR) if f.startswith('05c_'))))

                                  comprobacion      dif_max   tolerancia  coincide
    pasos: inercia, |r(PC1, t)| y congruencias 3.452315e-15 1.000000e-10      True
pasos: componentes retenidos por Horn (número) 0.000000e+00 0.000000e+00      True
                                      ganancia 4.440892e-16 1.000000e-14      True
    espectro: fracción en períodos >= 24 meses 1.332268e-15 1.000000e-10      True
            estacionalidad: ACF a 1 y 12 meses 2.442491e-15 1.000000e-10      True
Archivos escritos: 05c_espectro.csv, 05c_estacional.csv, 05c_ganancia.csv, 05c_pasos.csv
